In [ ]:
# Late-Payment Risk Model — 逾期付款风险模型
**Project by Joanne (Nn)** · Python, pandas, scikit-learn

Goal: predict which accounts-receivable invoices are likely to be paid late, so the AR team knows who to chase first.
目标：预测哪些应收账款发票可能逾期，让应收团队知道先联系谁。

**Before you start:** put `invoices.csv` in the same folder as this notebook. Run each box with **Shift + Enter**, from top to bottom.
开始前：把 `invoices.csv` 放在和这个笔记本相同的文件夹里。从上到下，每格按 **Shift + Enter** 运行。

## Lesson 1 — Load and look at the data · 第一课：读取并查看数据

`pandas` is the tool for tables in Python. `df` is our table (a "dataframe"), like an Excel sheet.
`pandas` 是Python处理表格的工具。`df` 是我们的表格，就像Excel工作表。

In [ ]:
import pandas as pd

df = pd.read_csv("invoices.csv")
df.head()

`df.shape` = (rows, columns). Should be (600, 8). · 行数和列数，应该是 (600, 8)。

In [ ]:
df.shape

`df.info()` shows each column's data type. Dates are still text (`object`) — we fix that in Lesson 2.
显示每列的数据类型。日期现在还是文字（`object`）——第二课修正。

In [ ]:
df.info()

How many invoices are late? · 有多少发票逾期？

In [ ]:
df["paid_late"].value_counts()

In [ ]:
df["paid_late"].value_counts(normalize=True) * 100

Late % per customer, worst first. · 每个客户的逾期率，最差的排前面。

In [ ]:
df.groupby("customer")["paid_late"].apply(lambda s: (s == "Yes").mean() * 100).sort_values(ascending=False)

**Finding 1 · 发现1:** 32.7% of invoices are late. Delta Foods (70%) and Iris Design (56%) are the worst; Evergreen Tech (8.6%) is the best.
32.7%的发票逾期。Delta Foods（70%）和 Iris Design（56%）最差；Evergreen Tech（8.6%）最好。

## Lesson 2 — Prepare the data · 第二课：准备数据

A model only understands numbers. We turn dates, Yes/No and customer names into numbers. This is called **feature engineering** (特征工程).
模型只懂数字。我们把日期、是/否和客户名都变成数字。

Convert text dates into real dates. · 把文字日期变成真正的日期。

In [ ]:
df["invoice_date"] = pd.to_datetime(df["invoice_date"])
df["due_date"] = pd.to_datetime(df["due_date"])
df.info()

Take the month out of the date, and flag December (year-end is busy, payments slip).
从日期里取出月份，并标记12月（年底忙，付款容易拖）。

In [ ]:
df["invoice_month"] = df["invoice_date"].dt.month
df["is_december"] = (df["invoice_month"] == 12).astype(int)
df[["invoice_date", "invoice_month", "is_december"]].head()

Target column: 1 = late, 0 = on time. This is what we want to predict.
目标列：1 = 逾期，0 = 准时。这是我们要预测的东西。

In [ ]:
df["late"] = (df["paid_late"] == "Yes").astype(int)
df[["paid_late", "late"]].head()

One-hot encoding (独热编码): one "customer" column becomes 10 columns of 1/0, one per customer.

We keep only the columns the model may use. We leave out `invoice_id` (just a label) and `payment_date` (that would be cheating — we don't know it yet when we predict).
只保留模型可以用的列。去掉 `invoice_id`（只是编号）和 `payment_date`（那是作弊——预测时还不知道付款日期）。

In [ ]:
df_model = pd.get_dummies(
    df[["customer", "amount", "payment_terms_days", "invoice_month", "is_december", "late"]],
    columns=["customer"], dtype=int
)
df_model.head()

In [ ]:
df_model.shape   # should be (600, 15)

## Lesson 3 — Train the model · 第三课：训练模型

Like training a new AR staff member: show them 480 old invoices with answers (**training set** 训练集), then test them on 120 invoices they have never seen (**test set** 测试集).

`X` = inputs (the columns the model can look at). `y` = the answer.
大写X = 输入，小写y = 答案。

In [ ]:
X = df_model.drop(columns=["late"])
y = df_model["late"]
print(X.shape, y.shape)

Split 80% training / 20% test. `random_state=42` = same random pick every time.
分成80%训练 / 20%测试。`random_state=42` = 每次挑同样的随机发票。

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

**Random Forest** (随机森林): builds many small decision trees, each votes late / not late, majority wins.
`model.fit(...)` is the training step. `max_depth=4` stops the trees from memorising the training data (**overfitting** 过拟合).
`model.fit(...)` 就是训练。`max_depth=4` 防止树死记训练数据（过拟合）。

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42)
model.fit(X_train, y_train)

Test on the 120 unseen invoices. · 用120张没见过的发票测试。

**How to read the number:** guessing "not late" for everything already gives 0.667 (the baseline 基准线). The model must beat that. With only 5 pieces of information per invoice, and noisy data, ~0.68 is expected. Real company data with more columns would score higher.
全部猜“不逾期”已经能得0.667（基准线）。模型必须超过它。只有5个信息、数据有噪音，~0.68是正常的。真实公司数据列更多，分数会更高。

In [ ]:
from sklearn.metrics import accuracy_score

predictions = model.predict(X_test)
accuracy_score(y_test, predictions)

## Lesson 4 — Make it useful: ranking and drivers · 第四课：让模型有用——排序和驱动因素

The AR team doesn't need a perfect yes/no. They need: *which invoices should I chase first?* So we ask the model for a **probability** (概率) and sort by it.
应收团队不需要完美的是/否，他们需要知道先追哪些发票。所以我们要模型给出概率，再排序。

In [ ]:
proba = model.predict_proba(X_test)[:, 1]

results = df.loc[X_test.index, ["invoice_id", "customer", "amount", "paid_late"]].copy()
results["risk_score"] = (proba * 100).round(0)
results.sort_values("risk_score", ascending=False).head(15)

Most of the top-15 should be "Yes" — calling these customers first catches mostly real late payers. That is the business value.
前15张大部分应该是“Yes”——先联系这些客户，抓到的大多是真正会逾期的。这就是业务价值。

**Feature importance** (特征重要性): which columns the model relied on most.

In [3]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

NameError: name 'pd' is not defined

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
importance.round(3)

**Finding 2 · 发现2:** invoice amount, Delta Foods, invoice month and Iris Design are the main drivers. The model found the risky customers by itself, and added that bigger invoices are more likely to be late.
发票金额、Delta Foods、月份和 Iris Design 是主要驱动因素。模型自己找到了高风险客户，还发现金额越大越容易逾期。

## Lesson 5 — Use it on new invoices and save it · 第五课：用于新发票并保存

A new invoice arrives: Delta Foods, $15,000, 30-day terms, December. Will it be late?
`reindex` makes the columns match the training data exactly (the model is strict about column order).
`reindex` 让列和训练数据完全一致（模型对列的顺序非常严格）。

In [ ]:
new_invoice = pd.DataFrame([{
    "amount": 15000,
    "payment_terms_days": 30,
    "invoice_month": 12,
    "is_december": 1,
    "customer": "Delta Foods"
}])

new_invoice = pd.get_dummies(new_invoice, columns=["customer"], dtype=int)
new_invoice = new_invoice.reindex(columns=X.columns, fill_value=0)

risk = model.predict_proba(new_invoice)[0, 1]
print(f"Risk of late payment: {risk*100:.0f}%")

Try changing the customer to `Evergreen Tech` and the amount to `3000` — the risk should drop a lot.
试试把客户改成 `Evergreen Tech`、金额改成 `3000`——风险应该大幅下降。

**Save the model** with `joblib` (like Save / Open in Excel, but for a model). `dump` = save, `load` = open.
用 `joblib` 保存模型（像Excel的保存/打开）。`dump` = 保存，`load` = 读回。

In [ ]:
import joblib

joblib.dump(model, "late_payment_model.pkl")
joblib.dump(list(X.columns), "model_columns.pkl")
print("Saved.")

### Next week: load and use without re-training · 下周：读回来直接用，不用重新训练
This box works in a brand-new notebook — no `invoices.csv`, no `fit`.
这一格在全新的笔记本里也能运行——不需要 `invoices.csv`，不需要 `fit`。

In [ ]:
import pandas as pd
import joblib

model = joblib.load("late_payment_model.pkl")
columns = joblib.load("model_columns.pkl")

new_invoice = pd.DataFrame([{
    "amount": 3000,
    "payment_terms_days": 60,
    "invoice_month": 5,
    "is_december": 0,
    "customer": "Evergreen Tech"
}])
new_invoice = pd.get_dummies(new_invoice, columns=["customer"], dtype=int)
new_invoice = new_invoice.reindex(columns=columns, fill_value=0)

risk = model.predict_proba(new_invoice)[0, 1]
print(f"Risk of late payment: {risk*100:.0f}%")

In [2]:
import matplotlib.pyplot as plt

imp = importance.sort_values()
labels = [i.replace("customer_", "Customer: ")
           .replace("amount", "Invoice amount")
           .replace("invoice_month", "Invoice month")
           .replace("payment_terms_days", "Payment terms (days)")
           .replace("is_december", "December invoice") for i in imp.index]

fig, ax = plt.subplots(figsize=(9, 6.2), dpi=200)
bars = ax.barh(labels, imp.values, color="#256abf", height=0.62)
for b, v in zip(bars, imp.values):
    ax.text(v + 0.004, b.get_y() + b.get_height()/2, f"{v:.3f}", va="center", fontsize=9, color="#444")
ax.set_xlim(0, 0.28)
for s in ["top", "right", "left"]:
    ax.spines[s].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.xaxis.grid(True, color="#e6e6e3"); ax.set_axisbelow(True)
ax.set_xlabel("Feature importance (share of the model's decisions, sums to 1.0)", fontsize=9.5, color="#444")
fig.text(0.02, 0.955, "Late Payment Model: Feature Importance", fontsize=15, fontweight="bold")
fig.text(0.02, 0.915, "What drives late payment? Random Forest on 600 synthetic AR invoices from 10 customers", fontsize=10, color="#555")
fig.text(0.02, 0.012, "Practice data (synthetic). Importance is not accuracy or probability — it shows which inputs the model relied on.", fontsize=8, color="#777")
plt.tight_layout(rect=[0, 0.035, 1, 0.90])
fig.savefig("feature_importance.png", facecolor="white")
plt.show()

NameError: name 'importance' is not defined

In [ ]:
## Resume / LinkedIn wording · 简历用语

> **Late-Payment Risk Model (personal project, Python)** — Built a machine learning model to rank accounts-receivable invoices by risk of late payment. Cleaned and engineered features with pandas, trained and tuned a Random Forest classifier with scikit-learn, and identified invoice size and customer history as the main drivers of late payment. Output: a risk-scored invoice list to prioritise collection calls.

## Key words to remember · 要记住的关键词
- **feature / feature engineering** 特征 / 特征工程 — the input columns, and how we build them
- **target** 目标 — the column we predict (`late`)
- **one-hot encoding** 独热编码 — turning categories into 1/0 columns
- **training set / test set** 训练集 / 测试集 — learn on one, check on the other
- **baseline** 基准线 — the score you get by guessing the majority; the model must beat it
- **overfitting** 过拟合 — the model memorises instead of learning; fix with simpler trees
- **accuracy** 准确率 — % of correct guesses
- **predict_proba** — probability instead of yes/no; use it to rank
- **feature importance** 特征重要性 — which inputs the model relied on